In [0]:
# =====================================================
# TELCO - PREDICCION AUTOMATIZADA DE SEGMENTOS
# =====================================================

import mlflow
import mlflow.spark
from pyspark.sql import functions as F

# 1. Configuración
tabla_origen = "workspace.telco_silver.clientes"

model_uri = (
    "runs:/3ecf204d49f6496dab452287808848db/"
    "modelo_telco_kmeans_k5"
)

ruta_temporal = "/Volumes/workspace/telco_ml/mlflow_tmp"

# 2. Cargar clientes desde Silver
df_clientes = spark.table(tabla_origen)

# 3. Preparar variables como en el entrenamiento
df_entrada = (
    df_clientes
    .withColumn("fecha_alta", F.to_date("fecha_alta"))
    .withColumn(
        "antiguedad_meses",
        F.floor(
            F.months_between(
                F.current_date(),
                F.col("fecha_alta")
            )
        )
    )
    .filter(
        F.col("ciudad").isNotNull()
        & F.col("segmento").isNotNull()
        & F.col("antiguedad_meses").isNotNull()
        & (F.col("antiguedad_meses") >= 0)
    )
    .select(
        "cliente_id",
        "ciudad",
        "segmento",
        "antiguedad_meses"
    )
)

# 4. Recuperar modelo desde MLflow
modelo = mlflow.spark.load_model(
    model_uri=model_uri,
    dfs_tmpdir=ruta_temporal
)

# 5. Generar predicciones
df_predicciones = modelo.transform(df_entrada)

# 6. Validaciones
total_entrada = df_entrada.count()
total_predicciones = df_predicciones.count()

assert total_entrada == total_predicciones

assert df_predicciones.filter(
    F.col("cluster").isNull()
).count() == 0

print("✅ PREDICCIONES GENERADAS CORRECTAMENTE")
print("Clientes procesados:", total_predicciones)

display(
    df_predicciones
    .groupBy("cluster")
    .count()
    .orderBy("cluster")
)

In [0]:
# ==========================================================
# TELCO - ACTUALIZACION AUTOMATICA DE SEGMENTOS
# ==========================================================

from pyspark.sql import functions as F

tabla_destino = "workspace.telco_ml.clientes_segmentados"

# 1. Preparar resultados
df_actualizado = (
    df_predicciones
    .select(
        "cliente_id",
        "ciudad",
        "segmento",
        "antiguedad_meses",
        "cluster"
    )
    .withColumn("modelo_k", F.lit(5))
    .withColumn(
        "fecha_segmentacion",
        F.current_timestamp()
    )
)

# 2. Validar datos
total = df_actualizado.count()

total_silver = spark.table(
    "workspace.telco_silver.clientes"
).count()

assert total > 0, "No hay clientes para procesar"

assert total == total_silver, (
    "Hay clientes de Silver sin segmentar"
)

assert df_actualizado.filter(
    F.col("cliente_id").isNull()
    | F.col("cluster").isNull()
).count() == 0, "Existen registros invalidos"

assert df_actualizado.select(
    "cliente_id"
).distinct().count() == total, "Hay clientes duplicados"

# 3. Crear vista temporal
df_actualizado.createOrReplaceTempView(
    "tmp_segmentacion_ml"
)

# 4. Actualizar tabla Delta
spark.sql(f"""
MERGE INTO {tabla_destino} AS t
USING tmp_segmentacion_ml AS s
ON t.cliente_id = s.cliente_id

WHEN MATCHED AND NOT (
    t.ciudad <=> s.ciudad
    AND t.segmento <=> s.segmento
    AND t.antiguedad_meses <=> s.antiguedad_meses
    AND t.cluster <=> s.cluster
    AND t.modelo_k <=> s.modelo_k
)
THEN UPDATE SET
    t.ciudad = s.ciudad,
    t.segmento = s.segmento,
    t.antiguedad_meses = s.antiguedad_meses,
    t.cluster = s.cluster,
    t.modelo_k = s.modelo_k,
    t.fecha_segmentacion = s.fecha_segmentacion

WHEN NOT MATCHED THEN INSERT (
    cliente_id,
    ciudad,
    segmento,
    antiguedad_meses,
    cluster,
    modelo_k,
    fecha_segmentacion
)
VALUES (
    s.cliente_id,
    s.ciudad,
    s.segmento,
    s.antiguedad_meses,
    s.cluster,
    s.modelo_k,
    s.fecha_segmentacion
)
""")

# 5. Validar resultados
df_final = spark.table(tabla_destino)

assert df_final.count() == total

print("✅ SEGMENTACION ACTUALIZADA CORRECTAMENTE")
print("Total clientes:", df_final.count())

display(
    df_final.groupBy("cluster")
    .count()
    .orderBy("cluster")
)